# 🎬 ImageEffectStudio — UI Version

Run **both cells** (Shift+Enter twice). The second cell launches an interactive control panel.

## Step 1 — Install dependencies

In [ ]:
!pip install opencv-python-headless tqdm numpy ipywidgets -q

## Step 2 — Launch the control panel
Set your options, then click **▶ Upload & Render**.

In [ ]:
# ============================================================
# ImageEffectStudio — UI Version
# All modules are embedded here — no separate .py files needed.
# ============================================================
import sys, types, importlib

# ── Embed config as a live module ───────────────────────────
cfg = types.ModuleType("config")
sys.modules["config"] = cfg

# ── Utils ───────────────────────────────────────────────────
import cv2
import numpy as np


def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Cannot load image: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32)


def resize_keep_aspect(img, target_w, target_h):
    h, w = img.shape[:2]
    scale = min(target_w / w, target_h / h)
    new_w = int(w * scale)
    new_h = int(h * scale)
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)


def center_on_canvas(img, canvas_w, canvas_h, bg_color=(0, 0, 0)):
    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.float32)
    canvas[:] = bg_color
    h, w = img.shape[:2]
    x_offset = (canvas_w - w) // 2
    y_offset = (canvas_h - h) // 2
    canvas[y_offset:y_offset + h, x_offset:x_offset + w] = img
    return canvas


def normalize_frame(frame):
    return np.clip(frame, 0, 255).astype(np.uint8)


def linear_interpolate(a, b, t):
    return a + (b - a) * t


def smoothstep(t):
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3 - 2 * t)


def in_range(t, start, end):
    return start <= t <= end


def effect_progress(t, start, end, smooth=True):
    if t < start or t > end:
        return 0.0
    progress = (t - start) / (end - start)
    return smoothstep(progress) if smooth else progress


# ── Animation ───────────────────────────────────────────────
import numpy as np
# (smoothstep already defined above)


# ── Easing functions ────────────────────────────────────────

def linear(t):          return t
def ease_in_quad(t):    return t * t
def ease_out_quad(t):   return t * (2 - t)

def ease_in_out_quad(t):
    # FIX 1: correct standard ease-in-out-quad
    return 2*t*t if t < 0.5 else -1 + (4 - 2*t)*t

def ease_in_cubic(t):   return t ** 3
def ease_out_cubic(t):  return 1 - (1 - t) ** 3

def ease_in_out_cubic(t):
    # FIX 2: correct cubic (not the quintic that was here before)
    return 4*t*t*t if t < 0.5 else 1 - (-2*t + 2)**3 / 2

def smooth(t):          return smoothstep(t)
def bounce(t):          return abs(np.sin(6.28 * (t + 1) * (1 - t)))
def elastic(t):         return np.sin(-13*(t+1)*np.pi/2) * (2**(-10*t)) + 1

def back(t):
    c1, c3 = 1.7, 2.7
    return c3 * t**3 - c1 * t**2

def expo_in(t):   return 0.0 if t == 0 else 2 ** (10*(t-1))
def expo_out(t):  return 1.0 if t == 1 else 1 - 2**(-10*t)
def sine(t):      return np.sin(t * np.pi / 2)
def pulse(t):     return 0.5 + 0.5 * np.sin(6.28 * t)


# ── Keyframe interpolation ───────────────────────────────────

def keyframe(t, keys):
    """
    FIX 3: now correctly returns the FIRST value when t is before the
    first keyframe (was falling through and returning the last value).
    keys = [(time, value), ...]  sorted ascending by time.
    """
    if t <= keys[0][0]:          # FIX 3
        return keys[0][1]
    for i in range(len(keys) - 1):
        t0, v0 = keys[i]
        t1, v1 = keys[i + 1]
        if t0 <= t <= t1:
            lt = smoothstep((t - t0) / (t1 - t0))
            return v0 + (v1 - v0) * lt
    return keys[-1][1]


# ── Animate helper ───────────────────────────────────────────

def animate(t, duration, style="smooth"):
    t = max(0.0, min(1.0, t / duration))
    curves = {
        "linear": linear, "ease_in": ease_in_quad, "ease_out": ease_out_quad,
        "ease_in_out": ease_in_out_quad, "cubic": ease_in_out_cubic,
        "smooth": smooth, "bounce": bounce, "elastic": elastic,
        "back": back, "expo_in": expo_in, "expo_out": expo_out,
        "sine": sine, "pulse": pulse,
    }
    return curves.get(style, smooth)(t)


# ── Animation presets ────────────────────────────────────────

def animation_preset(name, t):
    t = max(0.0, min(1.0, t))
    presets = {
        # Basic motion
        "fade_in": t, "fade_out": 1-t, "fade_in_out": 1-abs(2*t-1),
        "zoom_in": 1+0.3*t, "zoom_out": 1.3-0.3*t,
        "slide_left": -t, "slide_right": t, "slide_up": -t, "slide_down": t,
        "rotate": t*360, "shake": np.sin(t*50),
        "pulse": 1+0.1*np.sin(t*10), "breathing": 1+0.05*np.sin(t*2*np.pi),
        "blink": 1 if int(t*10)%2==0 else 0, "flash": 1 if t<0.1 else 0,
        "wave": np.sin(t*6.28), "wiggle": np.sin(t*20)*np.cos(t*10),
        "float": np.sin(t*3), "zoom_pulse": 1+0.2*np.sin(t*6.28),
        "camera_drift": np.sin(t*2), "glitch_pulse": np.random.rand()*t,
        "strobe": 1 if np.sin(t*50)>0 else 0,
        # Cinematic
        "cinematic_zoom": 1+0.15*smoothstep(t),
        "slow_zoom_in": 1+0.2*ease_in_out_cubic(t),
        "slow_zoom_out": 1.2-0.2*ease_in_out_cubic(t),
        "pan_left": -0.5*smoothstep(t), "pan_right": 0.5*smoothstep(t),
        "tilt": np.sin(t*np.pi), "dolly_in": 1+t*0.25, "dolly_out": 1.25-t*0.25,
        "orbit": np.sin(t*6.28), "orbit_reverse": np.cos(t*6.28),
        "focus_pulse": 1+0.05*np.sin(t*20), "depth_breath": 1+0.08*np.sin(t*3),
        "cinematic_fade": smoothstep(t), "dramatic_pause": 0 if t<0.5 else 1,
        # Glitch
        "glitch_jitter": np.random.normal(0, 0.1),
        "digital_warp": np.sin(t*20)*np.random.rand(),
        "signal_loss": 1 if np.random.rand()>t else 0,
        "data_corruption": np.random.rand()*np.sin(t*10),
        "frame_skip": int(t*10)%2, "rgb_shift_anim": np.sin(t*15),
        "scan_jitter": np.sin(t*60), "vhs_tracking": np.sin(t*8),
        "crt_roll": np.sin(t*4), "noise_burst": np.random.rand()*(1-t),
        "compression_artifact": np.random.normal(0, 1),
        "pixel_stretch": 1+np.sin(t*10)*0.1, "wave_distort": np.sin(t*12),
        "frame_wobble": np.cos(t*25), "signal_noise": np.random.randn(),
        "glitch_breath": 1+0.2*np.sin(t*10), "random_flash": np.random.rand(),
        "data_stream": t*np.random.rand(), "quantize": int(t*10)/10,
        "final_distort": np.sin(t*30)*np.random.rand(),
    }
    return presets.get(name, t)


# ── Effects ─────────────────────────────────────────────────
import cv2
import numpy as np
import random
# (effect_progress already defined above)


def apply_effect(frame, t, config):
    effect = config.EFFECT.lower()
    dispatch = {
        "glitch":               lambda: glitch(frame, t, config),
        "gentleripple":         lambda: gentle_ripple(frame, t, config),
        "chromaticaberration":  lambda: chromatic_aberration(frame, config.RGB_SHIFT),
        "heatwave":             lambda: heat_wave(frame, t),
        "dreamblur":            lambda: dream_blur(frame, t),
        "zoompulse":            lambda: zoom_pulse(frame, t, config),
        "camerashake":          lambda: camera_shake(frame, t),
        "float":                lambda: float_effect(frame, t),
        "breathingzoom":        lambda: breathing_zoom(frame, t),
        "wavewarp":             lambda: wave_warp(frame, t),
        "filmgrain":            lambda: film_grain(frame, config.NOISE_STRENGTH),
        "vhs":                  lambda: vhs(frame, t),
        "crt":                  lambda: crt(frame),
        "rgbsplit":             lambda: rgb_split(frame),
        "scanlines":            lambda: scanlines(frame),
        "bloom":                lambda: bloom(frame),
        "pixelsort":            lambda: pixel_sort(frame),
        "lightleak":            lambda: light_leak(frame, t),
        "digitaldistortion":    lambda: digital_distortion(frame, t),
    }
    return dispatch[effect]() if effect in dispatch else frame


# ── 1. Glitch ───────────────────────────────────────────────
def glitch(frame, t, config):
    h, w = frame.shape[:2]
    if random.random() < config.GLITCH_PROBABILITY:
        y         = random.randint(0, h - 30)
        slice_h   = random.randint(config.GLITCH_SLICE_MIN, config.GLITCH_SLICE_MAX)
        shift     = random.randint(-config.GLITCH_SHIFT_MAX, config.GLITCH_SHIFT_MAX)
        frame[y:y + slice_h] = np.roll(frame[y:y + slice_h], shift, axis=1)
    noise = np.random.normal(0, 5, frame.shape)
    return np.clip(frame + noise, 0, 255)


# ── 2. Gentle Ripple  (FIX 6 — vectorised) ─────────────────
def gentle_ripple(frame, t, config):
    h, w   = frame.shape[:2]
    rows   = np.arange(h, dtype=np.float32)
    shifts = (config.RIPPLE_STRENGTH *
              np.sin(2 * np.pi * (rows / 120.0 * config.RIPPLE_FREQUENCY
                                  + t * config.RIPPLE_SPEED))).astype(np.int32)
    out = np.empty_like(frame)
    for y, s in enumerate(shifts):
        out[y] = np.roll(frame[y], s, axis=0)
    return out


# ── 3. Chromatic Aberration  (FIX 4 — correct RGB split order) ─
def chromatic_aberration(frame, shift):
    # frame is RGB; cv2.split yields ch0=R, ch1=G, ch2=B
    r, g, b = cv2.split(frame)   # FIX 4: was b,g,r — mislabelled
    r = np.roll(r,  shift, axis=1)
    b = np.roll(b, -shift, axis=1)
    return cv2.merge([r, g, b])  # FIX 4: merge back in RGB order


# ── 4. Heat Wave  (FIX 7 — vectorised) ─────────────────────
def heat_wave(frame, t):
    h, w   = frame.shape[:2]
    rows   = np.arange(h, dtype=np.float32)
    shifts = (5 * np.sin(rows * 0.05 + t * 3)).astype(np.int32)
    out    = np.empty_like(frame)
    for y, s in enumerate(shifts):
        out[y] = np.roll(frame[y], s, axis=1)
    return out


# ── 5. Dream Blur ───────────────────────────────────────────
def dream_blur(frame, t):
    return cv2.GaussianBlur(frame, (9, 9), 2)


# ── 6. Zoom Pulse ───────────────────────────────────────────
def zoom_pulse(frame, t, config):
    h, w = frame.shape[:2]
    scale = 1.0 + 0.05 * np.sin(t * 2)
    nw, nh = int(w * scale), int(h * scale)
    resized = cv2.resize(frame, (nw, nh))
    x = (nw - w) // 2
    y = (nh - h) // 2
    return resized[y:y + h, x:x + w]


# ── 7. Camera Shake ─────────────────────────────────────────
def camera_shake(frame, t):
    dx = int(np.random.randint(-3, 3))
    dy = int(np.random.randint(-3, 3))
    return np.roll(np.roll(frame, dx, axis=1), dy, axis=0)


# ── 8. Float ────────────────────────────────────────────────
def float_effect(frame, t):
    return np.roll(frame, int(2 * np.sin(t)), axis=0)


# ── 9. Breathing Zoom  (FIX 9 — proper config class) ────────
class _BreathConfig:
    ZOOM_MIN = 1.0
    ZOOM_MAX = 1.1

def breathing_zoom(frame, t):
    return zoom_pulse(frame, t * 0.5, _BreathConfig())


# ── 10. Wave Warp  (FIX 8 — vectorised) ────────────────────
def wave_warp(frame, t):
    h, w   = frame.shape[:2]
    rows   = np.arange(h, dtype=np.float32)
    shifts = (10 * np.sin(rows * 0.03 + t * 2)).astype(np.int32)
    out    = np.empty_like(frame)
    for y, s in enumerate(shifts):
        out[y] = np.roll(frame[y], s, axis=0)
    return out


# ── 11. Film Grain ──────────────────────────────────────────
def film_grain(frame, strength):
    noise = np.random.normal(0, strength, frame.shape)
    return np.clip(frame + noise, 0, 255)


# ── 12. VHS ─────────────────────────────────────────────────
def vhs(frame, t):
    h, w = frame.shape[:2]
    frame = np.roll(frame, int(np.sin(t * 5) * 2), axis=1)
    if random.random() < 0.1:
        y = random.randint(0, h - 20)
        frame[y:y + 10] = np.roll(frame[y:y + 10], 10, axis=1)
    return frame


# ── 13. CRT ─────────────────────────────────────────────────
def crt(frame):
    out = frame.copy()
    out[::2] = out[::2] * 0.85
    return out


# ── 14. RGB Split  (FIX 5 — correct RGB split order) ────────
def rgb_split(frame):
    r, g, b = cv2.split(frame)   # FIX 5: was b,g,r — mislabelled
    b = np.roll(b,  2, axis=0)
    r = np.roll(r, -2, axis=0)
    return cv2.merge([r, g, b])  # FIX 5: merge back in RGB order


# ── 15. Scanlines  (FIX 10 — no in-place mutation of input) ─
def scanlines(frame):
    out = frame.copy()           # FIX 10: was mutating frame in-place
    out[::4] = out[::4] * 0.6
    return out


# ── 16. Bloom ───────────────────────────────────────────────
def bloom(frame):
    blur = cv2.GaussianBlur(frame, (15, 15), 5)
    return np.clip(frame * 0.8 + blur * 0.3, 0, 255)


# ── 17. Pixel Sort ──────────────────────────────────────────
def pixel_sort(frame):
    out = frame.copy()
    h   = out.shape[0]
    for y in range(0, h, 20):
        out[y:y + 20] = np.sort(out[y:y + 20], axis=1)
    return out


# ── 18. Light Leak ──────────────────────────────────────────
def light_leak(frame, t):
    h, w    = frame.shape[:2]
    overlay = frame.copy()
    x       = int((np.sin(t) + 1) * w / 2)
    cv2.circle(overlay, (x, h // 2), 200, (255, 180, 100), -1)
    return np.clip(frame * 0.7 + overlay * 0.3, 0, 255)


# ── 19. Digital Distortion  (FIX 11 — explicit float cast) ──
def digital_distortion(frame, t):
    noise = np.random.randint(-10, 10, frame.shape).astype(np.float32)  # FIX 11
    return np.clip(frame + noise, 0, 255)


# ── Renderer (no file I/O imports) ──────────────────────────
import cv2, numpy as np
from tqdm import tqdm

def _draw_text(frame, text):
    cv2.putText(frame, text, (50, 100),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2, cv2.LINE_AA)
    return frame

def _render(image_path):
    img        = load_image(image_path)
    img        = resize_keep_aspect(img, cfg.VIDEO_WIDTH, cfg.VIDEO_HEIGHT)
    base_frame = center_on_canvas(img, cfg.VIDEO_WIDTH, cfg.VIDEO_HEIGHT, cfg.BACKGROUND_COLOR)

    fourcc = cv2.VideoWriter_fourcc(*cfg.CODEC)
    out    = cv2.VideoWriter(cfg.OUTPUT_FILE, fourcc, cfg.FPS,
                             (cfg.VIDEO_WIDTH, cfg.VIDEO_HEIGHT))

    total_frames = int(cfg.FPS * cfg.VIDEO_DURATION)
    print(f"\n⚙️  Rendering {total_frames} frames  |  Effect: {cfg.EFFECT}")

    for i in tqdm(range(total_frames)):
        t     = i / cfg.FPS
        frame = base_frame.copy()
        if cfg.EFFECT_START_TIME <= t <= cfg.EFFECT_END_TIME:
            frame = apply_effect(frame, t, cfg)
        frame = normalize_frame(frame)
        frame = _draw_text(frame, cfg.EFFECT)
        out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

    out.release()
    print(f"✅  Done → {cfg.OUTPUT_FILE}")

# ── ipywidgets UI ───────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import files as colab_files

EFFECTS_LIST = [
    "Glitch", "GentleRipple", "ChromaticAberration", "HeatWave",
    "DreamBlur", "ZoomPulse", "CameraShake", "Float", "BreathingZoom",
    "WaveWarp", "FilmGrain", "VHS", "CRT", "RGBSplit", "Scanlines",
    "Bloom", "PixelSort", "LightLeak", "DigitalDistortion",
]

style = {"description_width": "160px"}
layout = widgets.Layout(width="420px")

w_effect    = widgets.Dropdown(options=EFFECTS_LIST, value="Glitch",
                               description="Effect:", style=style, layout=layout)
w_duration  = widgets.IntSlider(value=8, min=2, max=30, step=1,
                                description="Duration (s):", style=style, layout=layout)
w_fps       = widgets.Dropdown(options=[24, 30, 60], value=30,
                               description="FPS:", style=style, layout=layout)
w_width     = widgets.Dropdown(options=[720, 1080, 1440], value=1080,
                               description="Width (px):", style=style, layout=layout)
w_height    = widgets.Dropdown(options=[1280, 1920, 2560], value=1920,
                               description="Height (px):", style=style, layout=layout)
w_start     = widgets.FloatSlider(value=1.0, min=0.0, max=29.0, step=0.5,
                                  description="Effect start (s):", style=style, layout=layout)
w_end       = widgets.FloatSlider(value=7.0, min=1.0, max=30.0, step=0.5,
                                  description="Effect end (s):", style=style, layout=layout)
w_intensity = widgets.FloatSlider(value=0.8, min=0.0, max=1.0, step=0.05,
                                  description="Intensity:", style=style, layout=layout)
w_glitch_p  = widgets.FloatSlider(value=0.18, min=0.0, max=1.0, step=0.01,
                                  description="Glitch prob:", style=style, layout=layout)
w_ripple_s  = widgets.IntSlider(value=6, min=1, max=30,
                                description="Ripple strength:", style=style, layout=layout)
w_rgb_shift = widgets.IntSlider(value=5, min=1, max=30,
                                description="RGB shift:", style=style, layout=layout)
w_noise     = widgets.IntSlider(value=4, min=1, max=30,
                                description="Noise strength:", style=style, layout=layout)
w_output    = widgets.Text(value="output.mp4", description="Output file:", style=style, layout=layout)

w_audio_toggle = widgets.ToggleButtons(
    options=["No", "Yes"],
    value="No",
    description="Add background sound?",
    style={"description_width": "180px", "button_width": "70px"},
    layout=widgets.Layout(width="420px"),
)
w_audio_box = widgets.VBox([], layout=widgets.Layout(margin="0px"))

w_audio_volume = widgets.FloatSlider(value=0.8, min=0.0, max=1.0, step=0.05,
    description="Audio volume:", style=style, layout=layout)
w_audio_start  = widgets.FloatSlider(value=0.0, min=0.0, max=60.0, step=0.5,
    description="Audio start (s):", style=style, layout=layout)
w_audio_fade   = widgets.Checkbox(value=True, description="Fade audio out at end",
    style=style, layout=layout)
w_audio_note   = widgets.HTML(
    value="<span style=\'color:#888;font-size:12px\'>"
          "Supported formats: MP3, WAV, OGG, AAC · "
          "File will be uploaded when you click Render</span>"
)

def _toggle_audio(change):
    if change["new"] == "Yes":
        w_audio_box.children = [
            widgets.HTML("<b>── Background Sound ────────────────</b>"),
            w_audio_volume, w_audio_start, w_audio_fade, w_audio_note,
        ]
    else:
        w_audio_box.children = []

w_audio_toggle.observe(_toggle_audio, names="value")

w_btn       = widgets.Button(description="▶  Upload & Render",
                             button_style="success",
                             layout=widgets.Layout(width="200px", height="40px"))
w_status    = widgets.Output()

def on_render(_):
    w_status.clear_output()
    with w_status:
        # Apply widget values to cfg module
        cfg.EFFECT            = w_effect.value
        cfg.VIDEO_WIDTH       = w_width.value
        cfg.VIDEO_HEIGHT      = w_height.value
        cfg.FPS               = w_fps.value
        cfg.VIDEO_DURATION    = w_duration.value
        cfg.OUTPUT_FILE       = w_output.value
        cfg.BACKGROUND_COLOR  = (0, 0, 0)
        cfg.EFFECT_START_TIME = w_start.value
        cfg.EFFECT_END_TIME   = w_end.value
        cfg.INTENSITY         = w_intensity.value
        cfg.SMOOTHNESS        = 0.5
        cfg.GLITCH_PROBABILITY = w_glitch_p.value
        cfg.GLITCH_SLICE_MIN  = 10
        cfg.GLITCH_SLICE_MAX  = 40
        cfg.GLITCH_SHIFT_MAX  = 25
        cfg.RIPPLE_STRENGTH   = w_ripple_s.value
        cfg.RIPPLE_FREQUENCY  = 2.0
        cfg.RIPPLE_SPEED      = 2.0
        cfg.RGB_SHIFT         = w_rgb_shift.value
        cfg.NOISE_STRENGTH    = w_noise.value
        cfg.MOTION_BLUR_STRENGTH = 6
        cfg.ZOOM_MIN          = 1.0
        cfg.ZOOM_MAX          = 1.15
        cfg.CODEC             = "mp4v"
        cfg.BITRATE           = "12M"

        print("📂  Upload your image...")
        uploaded = colab_files.upload()
        if not uploaded:
            print("❌  No file uploaded.")
            return
        image_path = list(uploaded.keys())[0]
        _render(image_path)

        # ── Background sound ────────────────────────────────────
        if w_audio_toggle.value == "Yes":
            try:
                import subprocess, shutil, os
                # check ffmpeg available
                if shutil.which("ffmpeg") is None:
                    print("⚙️  Installing ffmpeg…")
                    subprocess.run(["apt-get", "install", "-y", "-q", "ffmpeg"],
                                   check=True, capture_output=True)

                print("🎵  Upload your audio file (MP3 / WAV / OGG / AAC)…")
                audio_up = colab_files.upload()
                if not audio_up:
                    print("⚠️  No audio uploaded — video saved without sound.")
                else:
                    audio_path = list(audio_up.keys())[0]
                    vol        = w_audio_volume.value
                    a_start    = w_audio_start.value
                    fade_flag  = w_audio_fade.value
                    video_dur  = cfg.VIDEO_DURATION

                    out_with_audio = "output_with_audio.mp4"

                    # Build ffmpeg audio filter
                    af = f"adelay={int(a_start*1000)}|{int(a_start*1000)},volume={vol}"
                    if fade_flag:
                        fade_start = max(0, video_dur - 2)
                        af += f",afade=t=out:st={fade_start}:d=2"

                    cmd = [
                        "ffmpeg", "-y",
                        "-i", cfg.OUTPUT_FILE,       # video (no audio)
                        "-i", audio_path,            # audio track
                        "-filter_complex", f"[1:a]{af}[aout]",
                        "-map", "0:v",
                        "-map", "[aout]",
                        "-c:v", "copy",
                        "-c:a", "aac", "-b:a", "192k",
                        "-shortest",
                        out_with_audio,
                    ]
                    print("🔧  Merging audio + video with ffmpeg…")
                    result = subprocess.run(cmd, capture_output=True, text=True)
                    if result.returncode != 0:
                        print("❌  ffmpeg error:")
                        print(result.stderr[-800:])
                    else:
                        os.replace(out_with_audio, cfg.OUTPUT_FILE)
                        print(f"✅  Audio merged → {cfg.OUTPUT_FILE}")
            except Exception as audio_err:
                print(f"⚠️  Audio merge failed: {audio_err}")
                print("    Video saved without sound.")

        colab_files.download(cfg.OUTPUT_FILE)

w_btn.on_click(on_render)

display(HTML("<h2 style=\'margin-bottom:12px\'>🎬 ImageEffectStudio</h2>"))
display(widgets.VBox([
    widgets.HTML("<b>── Video Settings ─────────────────</b>"),
    w_effect, w_width, w_height, w_fps, w_duration, w_output,
    widgets.HTML("<b>── Timing ──────────────────────────</b>"),
    w_start, w_end, w_intensity,
    widgets.HTML("<b>── Effect Parameters ───────────────</b>"),
    w_glitch_p, w_ripple_s, w_rgb_shift, w_noise,
    widgets.HTML("<b>── Background Sound ────────────────</b>"),
    w_audio_toggle,
    w_audio_box,
    widgets.HTML("<br>"),
    w_btn,
    w_status,
]))
